# NOTEBOOK 15 bis · Seq2seq sin atención

## Traducción inglés → español con un encoder y un decoder LSTM

El objetivo es construir una **referencia sin atención**, que después podamos comparar con el mismo modelo incorporando atención. No buscamos un traductor de uso general.

El encoder lee la frase completa. Sus estados finales `(hidden, cell)` inicializan el decoder. A partir de ahí, el decoder solo recibe la palabra anterior y sus propios estados: **no consulta los estados de todas las posiciones del encoder**.

Usaremos un vocabulario construido exclusivamente con entrenamiento y embeddings aprendidos desde cero. Esto evita descargar FastText, mantener dos matrices incompatibles y predecir entre casi un millón de palabras. Al comparar con atención, conservaremos los mismos datos, vocabularios y dimensiones.

**Requisitos:** Python, PyTorch y Matplotlib. Ejecuta las celdas en orden desde el directorio de este notebook. No se descarga ningún corpus automáticamente.

In [ ]:
from pathlib import Path
from collections import Counter
from itertools import zip_longest
import math
import random
import re
import time

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# CPU también funciona. Las GPU NVIDIA se usan si están disponibles.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

## 1. Datos y particiones

Leemos pares alineados de Europarl. Para una primera práctica limitamos la longitud y el número de ejemplos. El subconjunto son los primeros pares válidos del fichero; no es una muestra representativa de todo Europarl. Se pueden ampliar estos límites después.

Eliminamos pares duplicados y asignamos cada frase inglesa a una única partición. Así, una misma entrada no aparece en entrenamiento y prueba, aunque tenga varias traducciones. Esto no elimina paráfrasis ni repeticiones entre documentos; una evaluación más exigente separaría debates completos.

La tokenización separa palabras y signos de puntuación. Es deliberadamente sencilla; no utiliza subpalabras. Las palabras que no estén en el vocabulario se representarán mediante `<unk>`.

In [ ]:
SOURCE_FILE = Path("data/europarl/europarl-v7.es-en.en")
TARGET_FILE = Path("data/europarl/europarl-v7.es-en.es")
MAX_WORDS = 30             # Sin contar los tokens especiales.
MAX_PAIRS = 30_000         # Usa None para conservar todos los pares válidos.
MIN_FREQUENCY = 2
MAX_VOCAB_SIZE = 12_000    # Incluye los cuatro tokens especiales.


def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)


def read_pairs(source_file, target_file, max_words, max_pairs):
    pairs = []
    seen = set()
    with source_file.open(encoding="utf-8") as source, target_file.open(encoding="utf-8") as target:
        for line_number, (en, es) in enumerate(zip_longest(source, target), start=1):
            if en is None or es is None:
                raise ValueError(f"Ficheros desalineados en la línea {line_number}")
            en_tokens, es_tokens = tokenize(en), tokenize(es)
            if not (1 <= len(en_tokens) <= max_words and 1 <= len(es_tokens) <= max_words):
                continue
            pair = (tuple(en_tokens), tuple(es_tokens))
            if pair in seen:
                continue
            seen.add(pair)
            pairs.append(pair)
            if max_pairs is not None and len(pairs) >= max_pairs:
                break
    return pairs


def split_pairs(pairs, seed):
    sources = sorted({en for en, es in pairs})
    if len(sources) < 20:
        raise ValueError("Se necesitan al menos 20 entradas distintas para esta práctica.")
    random.Random(seed).shuffle(sources)
    n_valid = max(1, int(0.1 * len(sources)))
    n_test = max(1, int(0.1 * len(sources)))
    valid_sources = set(sources[:n_valid])
    test_sources = set(sources[n_valid:n_valid + n_test])
    train, valid, test = [], [], []
    for pair in pairs:
        en, es = pair
        if en in valid_sources:
            valid.append(pair)
        elif en in test_sources:
            test.append(pair)
        else:
            train.append(pair)
    return train, valid, test


pairs = read_pairs(SOURCE_FILE, TARGET_FILE, MAX_WORDS, MAX_PAIRS)
train_pairs, valid_pairs, test_pairs = split_pairs(pairs, SEED)
print("Pares de entrenamiento, validación y prueba:",
      len(train_pairs), len(valid_pairs), len(test_pairs))
print("Ejemplo:", train_pairs[0])

## 2. Vocabularios y lotes

Los identificadores especiales son iguales en ambos idiomas, pero cada idioma tiene su propio vocabulario y su propia capa de embeddings.

- `<pad>` rellena las secuencias de un lote.
- `<sos>` inicia la entrada del decoder.
- `<eos>` marca el final que debe aprender a predecir.
- `<unk>` representa palabras desconocidas.

No añadimos un `<pad>` extra a cada frase. El relleno se añade únicamente al formar el lote.

In [ ]:
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]


class Vocabulary:
    def __init__(self, sentences, min_frequency, max_size):
        counts = Counter(token for sentence in sentences for token in sentence)
        words = [word for word, count in counts.most_common()
                 if count >= min_frequency and word not in SPECIAL_TOKENS]
        self.itos = SPECIAL_TOKENS + words[:max_size - len(SPECIAL_TOKENS)]
        self.stoi = {word: index for index, word in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [self.stoi.get(token, UNK) for token in tokens]

    def decode(self, ids):
        words = []
        for index in ids:
            if index == EOS:
                break
            if index not in (PAD, SOS):
                words.append(self.itos[index])
        return words


src_vocab = Vocabulary((en for en, es in train_pairs), MIN_FREQUENCY, MAX_VOCAB_SIZE)
tgt_vocab = Vocabulary((es for en, es in train_pairs), MIN_FREQUENCY, MAX_VOCAB_SIZE)


class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.examples = []
        for en, es in pairs:
            source = torch.tensor(src_vocab.encode(en) + [EOS], dtype=torch.long)
            target = torch.tensor([SOS] + tgt_vocab.encode(es) + [EOS], dtype=torch.long)
            self.examples.append((source, target))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def collate_batch(examples):
    sources, targets = zip(*examples)
    source_lengths = torch.tensor([len(source) for source in sources])
    sources = pad_sequence(sources, batch_first=True, padding_value=PAD)
    targets = pad_sequence(targets, batch_first=True, padding_value=PAD)
    return sources, source_lengths, targets


BATCH_SIZE = 32
train_dataset = TranslationDataset(train_pairs, src_vocab, tgt_vocab)
valid_dataset = TranslationDataset(valid_pairs, src_vocab, tgt_vocab)
test_dataset = TranslationDataset(test_pairs, src_vocab, tgt_vocab)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_batch,
                          generator=torch.Generator().manual_seed(SEED))
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_batch)

print("Vocabularios inglés/español:", len(src_vocab), len(tgt_vocab))
sources, source_lengths, targets = next(iter(train_loader))
print("Entrada:", sources.shape, "Longitudes:", source_lengths.shape)
print("Objetivo con <sos> y <eos>:", targets.shape)

def unknown_rate(pairs, vocab, language_index):
    tokens = [token for pair in pairs for token in pair[language_index]]
    return sum(token not in vocab.stoi for token in tokens) / len(tokens)

print(f"Palabras desconocidas en prueba (EN): {unknown_rate(test_pairs, src_vocab, 0):.1%}")
print(f"Palabras desconocidas en prueba (ES): {unknown_rate(test_pairs, tgt_vocab, 1):.1%}")


## 3. Encoder: el último estado de cada capa

`pack_padded_sequence` indica las longitudes reales: el encoder no debe actualizar sus estados con los pasos de relleno.

Para una LSTM unidireccional, `hidden` y `cell` tienen forma `(n_capas, lote, dimensión_oculta)`. Pasamos **todas las capas** al decoder, que tiene la misma configuración. No hay atención ni acceso posterior a las salidas intermedias del encoder.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                            num_layers=num_layers, batch_first=True)

    def forward(self, source, source_lengths):
        embedded = self.embedding(source)
        packed = pack_padded_sequence(embedded, source_lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, (hidden, cell) = self.lstm(packed)
        return hidden, cell

## 4. Decoder: una palabra en cada paso

La entrada es un identificador por ejemplo, de forma `(lote,)`. La capa de embeddings produce su vector y añadimos una dimensión temporal de longitud uno.

La salida son **logits**, no probabilidades. `CrossEntropyLoss` aplica internamente la transformación necesaria. Para elegir la siguiente palabra basta con `argmax` sobre los logits.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                            num_layers=num_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids, hidden, cell):
        embedded = self.embedding(token_ids).unsqueeze(1)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        logits = self.output_layer(output[:, 0, :])
        return logits, hidden, cell

## 5. Seq2seq y teacher forcing

Para el objetivo `<sos> mi perro <eos>`, el primer paso recibe `<sos>` y debe predecir `mi`. Después recibe `mi` (teacher forcing) o su propia predicción. Nunca recibe la palabra que está intentando predecir en ese mismo paso.

La decisión de teacher forcing se toma por paso para todo el lote. En la rama sin teacher forcing realimentamos el **identificador predicho**, que pasa por la misma capa de embeddings utilizada con los tokens reales. `argmax` no es diferenciable; la pérdida de cada paso sí entrena sus logits y los estados recurrentes.

Durante el entrenamiento recorremos la longitud del objetivo del lote; los `<pad>` no contribuyen a la pérdida. La generación libre tendrá un método separado y no recibirá objetivos.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, source_lengths, target, teacher_forcing_ratio=0.5):
        hidden, cell = self.encoder(source, source_lengths)
        token_ids = target[:, 0]  # <sos>
        outputs = []

        for step in range(1, target.size(1)):
            logits, hidden, cell = self.decoder(token_ids, hidden, cell)
            outputs.append(logits)
            if random.random() < teacher_forcing_ratio:
                token_ids = target[:, step]
            else:
                token_ids = logits.argmax(dim=-1)

        # (lote, longitud_objetivo_sin_sos, vocabulario_español)
        return torch.stack(outputs, dim=1)

    @torch.no_grad()
    def generate(self, source, source_lengths, max_new_tokens=50):
        # El llamador debe activar model.eval(). No se utiliza la traducción real.
        hidden, cell = self.encoder(source, source_lengths)
        batch_size = source.size(0)
        token_ids = torch.full((batch_size,), SOS, dtype=torch.long, device=source.device)
        finished = torch.zeros(batch_size, dtype=torch.bool, device=source.device)
        predictions = []

        for _ in range(max_new_tokens):
            logits, hidden, cell = self.decoder(token_ids, hidden, cell)
            # <pad> y <sos> no son palabras que deban generarse.
            logits[:, PAD] = -torch.inf
            logits[:, SOS] = -torch.inf
            next_ids = logits.argmax(dim=-1)
            next_ids = torch.where(finished, PAD, next_ids)
            predictions.append(next_ids)
            finished = finished | (next_ids == EOS)
            token_ids = next_ids
            if finished.all().item():
                break

        return torch.stack(predictions, dim=1)


EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2

torch.manual_seed(SEED)
encoder = Encoder(len(src_vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS)
decoder = Decoder(len(tgt_vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS)
model = Seq2Seq(encoder, decoder).to(DEVICE)
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters()):,}")

## 6. Entrenamiento y selección mediante validación

En entrenamiento usamos teacher forcing parcial. En validación calculamos la pérdida con teacher forcing completo: mide la predicción del siguiente token dado el prefijo correcto y permite seleccionar el modelo sin aleatoriedad en la realimentación. **No mide por sí sola la calidad de la generación libre**, que evaluaremos después.

Sumamos la pérdida de los tokens válidos y la dividimos entre su número. Así, la pérdida media no depende directamente de cuánto relleno haya en un lote. Recortamos la norma de los gradientes para limitar actualizaciones inestables.

Una época recorre todo el conjunto de entrenamiento. El tiempo depende del equipo y del tamaño elegido; empieza con menos pares para comprobar el flujo.

In [ ]:
LEARNING_RATE = 0.001
NUM_EPOCHS = 15
TEACHER_FORCING_RATIO = 0.5

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_tokens = 0
    ratio = TEACHER_FORCING_RATIO if training else 1.0

    with torch.set_grad_enabled(training):
        for source, lengths, target in loader:
            source, target = source.to(DEVICE), target.to(DEVICE)
            if training:
                optimizer.zero_grad()

            logits = model(source, lengths, target, teacher_forcing_ratio=ratio)
            expected = target[:, 1:]  # No intentamos predecir <sos>.
            loss_sum = criterion(logits.reshape(-1, logits.size(-1)), expected.reshape(-1))
            n_tokens = (expected != PAD).sum().item()
            loss = loss_sum / n_tokens

            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss_sum.item()
            total_tokens += n_tokens

    return total_loss / total_tokens


history = {"train": [], "valid": []}
best_valid_loss = math.inf
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.perf_counter()
    train_loss = run_epoch(model, train_loader, optimizer)
    valid_loss = run_epoch(model, valid_loader)
    history["train"].append(train_loss)
    history["valid"].append(valid_loss)
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = {name: tensor.detach().cpu().clone()
                      for name, tensor in model.state_dict().items()}
    print(f"Época {epoch:02d} | entrenamiento {train_loss:.3f} | "
          f"validación {valid_loss:.3f} | {time.perf_counter() - start:.1f} s")

model.load_state_dict(best_state)
model.eval()

plt.plot(range(1, NUM_EPOCHS + 1), history["train"], label="Entrenamiento (TF parcial)")
plt.plot(range(1, NUM_EPOCHS + 1), history["valid"], label="Validación (TF completo)")
plt.xlabel("Época")
plt.ylabel("Entropía cruzada media por token")
plt.legend()
plt.show()

## 7. Traducción sin teacher forcing

Usamos búsqueda voraz: elegimos el token con mayor logit en cada paso. La generación termina con `<eos>` o al alcanzar el límite de seguridad. Para la comparación con atención utilizaremos exactamente el mismo procedimiento.

Una frase con palabras desconocidas pierde información antes de llegar al encoder. Conviene distinguir este problema de los errores que puede resolver la atención.

In [ ]:
@torch.no_grad()
def translate(sentence, model, max_new_tokens=50):
    model.eval()
    tokens = tokenize(sentence)
    source = torch.tensor([src_vocab.encode(tokens) + [EOS]], device=DEVICE)
    lengths = torch.tensor([source.size(1)])
    prediction = model.generate(source, lengths, max_new_tokens)
    words = tgt_vocab.decode(prediction[0].tolist())
    unknown = [token for token in tokens if token not in src_vocab.stoi]
    return " ".join(words), unknown


# Ejemplos fijos de prueba: no seleccionamos solo los que salen bien.
for en, es in test_pairs[:8]:
    prediction, unknown = translate(" ".join(en), model)
    print("EN:", " ".join(en))
    print("Referencia:", " ".join(es))
    print("Sin atención:", prediction)
    print("Palabras desconocidas:", unknown)
    print()

## 8. Evaluación de generación libre

Guardamos las predicciones de **toda la prueba**, con su longitud de entrada, para compararlas con el futuro modelo con atención. La coincidencia exacta es una medida muy estricta: una traducción válida puede ser distinta de la referencia. Se incluye como comprobación, no como única medida de calidad.

Opcionalmente, si instalas `sacrebleu`, calculamos BLEU sobre estos textos ya tokenizados y chrF. Al comparar modelos deben mantenerse las mismas referencias y opciones. Observa también omisiones, repeticiones y conservación de cifras; una mejora no tiene por qué aparecer en todas las frases.

In [ ]:
@torch.no_grad()
def evaluate_generation(model, loader, pairs):
    model.eval()
    records = []
    offset = 0
    for source, lengths, target in loader:
        predicted = model.generate(source.to(DEVICE), lengths, max_new_tokens=MAX_WORDS + 10)
        for ids in predicted.cpu().tolist():
            en, es = pairs[offset]
            hypothesis = " ".join(tgt_vocab.decode(ids))
            reference = " ".join(es)  # Referencia original, sin convertir palabras a <unk>.
            records.append({"source": " ".join(en), "reference": reference,
                            "prediction": hypothesis, "source_length": len(en)})
            offset += 1
    return records


test_results = evaluate_generation(model, test_loader, test_pairs)
for lower, upper in [(1, 10), (11, 20), (21, MAX_WORDS)]:
    group = [r for r in test_results if lower <= r["source_length"] <= upper]
    if group:
        exact = sum(r["prediction"] == r["reference"] for r in group) / len(group)
        print(f"Longitud {lower}–{upper}: {len(group)} frases; coincidencia exacta {exact:.1%}")

try:
    from sacrebleu.metrics import BLEU, CHRF
except ImportError:
    print("Métricas opcionales: instala sacrebleu para calcular BLEU y chrF.")
else:
    hypotheses = [r["prediction"] for r in test_results]
    references = [[r["reference"] for r in test_results]]
    for metric in (BLEU(tokenize="none"), CHRF()):
        print(metric.corpus_score(hypotheses, references))
        print("Configuración:", metric.get_signature())

## 9. Guardar una referencia reproducible

El checkpoint incluye pesos, vocabularios, configuración y particiones. El archivo de resultados conserva las traducciones sin atención. Para el siguiente experimento reutiliza estas particiones y vocabularios; no reconstruyas una división diferente.

Las semillas ayudan a repetir el experimento, aunque no garantizan resultados idénticos entre dispositivos y versiones. Para una conclusión más sólida, repite ambos entrenamientos con varias semillas.

Este checkpoint permite recuperar el mejor modelo para inferencia; no guarda el estado del optimizador para reanudar exactamente el entrenamiento.

In [ ]:
import json

ARTIFACT_DIR = Path("artifacts/nb05_bis")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "seed": SEED, "embedding_dim": EMBEDDING_DIM, "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS, "max_words": MAX_WORDS, "max_pairs": MAX_PAIRS,
    "min_frequency": MIN_FREQUENCY, "max_vocab_size": MAX_VOCAB_SIZE,
    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS, "teacher_forcing_ratio": TEACHER_FORCING_RATIO,
    "source_file": str(SOURCE_FILE), "target_file": str(TARGET_FILE),
    "torch_version": str(torch.__version__),
}
torch.save({"model_state": best_state, "config": config,
            "src_tokens": src_vocab.itos, "tgt_tokens": tgt_vocab.itos,
            "best_valid_loss": best_valid_loss}, ARTIFACT_DIR / "model.pt")

experiment = {"config": config, "src_tokens": src_vocab.itos, "tgt_tokens": tgt_vocab.itos,
              "train_pairs": train_pairs, "valid_pairs": valid_pairs, "test_pairs": test_pairs,
              "history": history, "test_results": test_results}
(ARTIFACT_DIR / "experiment.json").write_text(
    json.dumps(experiment, ensure_ascii=False, indent=2), encoding="utf-8")
print("Resultados guardados en", ARTIFACT_DIR.resolve())

### Qué comparar al incorporar atención

1. Conserva este encoder y decoder como referencia conceptual y mantén las particiones, vocabularios y protocolo de evaluación.
2. En el nuevo encoder, recupera también la secuencia de estados ocultos y enmascara el padding al calcular atención.
3. Permite que el decoder construya un contexto distinto en cada paso a partir de esos estados.
4. Compara las métricas globales, los resultados por longitud y las mismas frases de prueba. Visualiza también los pesos de atención.

La hipótesis es que consultar la entrada en cada paso ayudará a conservar información, especialmente en secuencias más largas. Los resultados deben comprobar esa hipótesis; no la damos por garantizada.